In [3]:
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

import matplotlib.pyplot as plt
from build_sam import sam_model_registry
import os

from glob import glob
import pandas as pd
from rich.progress import track

In [6]:
pannuke_metadata = pd.read_csv('../Datasets/Cell Datasets/PanNuke_multi_count/pannuke_meta.csv')
pannuke_metadata.head()

,image_path,num_cells
0,6369_Epithelial.png,12
1,123_Inflammatory.png,1
2,1426_Inflammatory.png,1
3,6884_Epithelial.png,19
4,4650_Inflammatory.png,25


In [7]:
pannuke = pannuke_metadata[pannuke_metadata['num_cells'] > 3]
pannuke.shape

(11101, 2)

In [3]:
img_resolution = 1024
sam = sam_model_registry['sam_encoder_h'](checkpoint='../segment-anything/checkpoints/sam_vit_h_image_encoder.pth', custom_img_size=img_resolution)
print(f"Loaded a SAM-H encoder")

Loaded encoder from checkpoint
Loaded a SAM-H encoder


In [14]:
def prepare_image(image, img_resolution=1024):
    image = np.array(image)
    image = torch.as_tensor(image).float()
    return image.permute(2, 0, 1)

In [16]:
os.makedirs('../Datasets/Cell Datasets/PanNuke_multi_count/embeddings', exist_ok=True)

In [ ]:
for i, row in track(pannuke.iterrows()):
    fname = row['image_path']
    img_path = os.path.join('../Datasets/Cell Datasets/PanNuke_multi_count/images/', fname)
    img = Image.open(img_path)
    img = prepare_image(img, img_resolution)
    img = img.unsqueeze(0)
    embeddings = sam(img)

    # save the embeddings to disk
    embedding_path = os.path.join('../Datasets/Cell Datasets/PanNuke_multi_count/embeddings/', fname.replace('.png', '.npy'))
    torch.save(embeddings, embedding_path)

    


    

In [19]:
# load the embeddings
embedding_files = glob('../Datasets/Cell Datasets/PanNuke_multi_count/embeddings/*.npy')

sample_embedding = torch.load(embedding_files[0])
print(sample_embedding.shape)

torch.Size([1, 256, 16, 16])


In [2]:
from glob import glob

embeddings = glob('../Datasets/Cell Datasets/PanNuke_multi_count/embeddings/*.npy')
print(len(embeddings))

11101


In [4]:
# Load each embedding and detach it from the graph
for emb_file in track(embeddings):
    emb = torch.load(emb_file, map_location='cpu')
    emb = emb.detach()

    # Save the embedding back to disk
    torch.save(emb, emb_file)
    

Output()